In [22]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import random # pour faire des tirages aléatoires

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [24]:
# 0) En Colab, habilite widgets
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [25]:
#%% Fonction utilisées dans le script
# Fonction de lecture des données
def read_excel_data(filename, sheet_name):
        data = pd.read_excel(filename, sheet_name=sheet_name, header=None)
        values =  data.values
        return values

# Modèle algébrique de production PV
def Mod_PVprod(Ta, E, Ppv_nom):

    # Paramètre du modèle
    NOCT = 48.
    E_ref = 1000. # Rayonnement de référence [W/m²]
    Tpv_ref = 25. # Température de référence [°C]
    alpha = -4.8e-3

    # à remplir ...
    # (2) Température des modules
    Tpv = Ta + E * (NOCT - 20.0) / 800.0

    # (1) Puissance PV
    Ppv = (E / E_ref) * (Ppv_nom + alpha * (Tpv - Tpv_ref))

    return Ppv

In [26]:
#%% Paramètres du problème
# Constantes :
deg2K  = 273.15 # Conversion degré en Kelvin
kW2W = 1000. # Conversion kW en W
kWh2Wh = 1000.
h2min = 60 # conversion h -> min
N_ev = 50
capa_bat = 52  # Capacité des batteries [kWh]
d_autonomie = 320 # Autonomie des batteries [km]
P1_ch = 22 # Puissance de recharge [kW] --> Hypothese: recharge normal
vmoy = 40/h2min # Vitesse moyenne en km par min

Ppv_nom = 500e3 # Puissance nominale de l'installation solaire

In [27]:
#%% Chargement des données
j = 174 # Jour de simulation
Nj = 7 # Nombre de jour de simu
ind_0 = j*24+1 # commence à 1h
ind_f = (j+Nj)*24 + 1 # termine à minuit

In [28]:
filename = 'TP-V2G-Input_data.xlsx'
sheet_name = "Demands"
Hd = np.array(read_excel_data(filename, sheet_name)[1+ind_0:1+ind_f,1].tolist())*kW2W
Pd = np.array(read_excel_data(filename, sheet_name)[1+ind_0:1+ind_f,2].tolist())*kW2W
v_tps_h = np.array(read_excel_data(filename, sheet_name)[1+ind_0:1+ind_f,0].tolist())*h2min

sheet_name = "Meteo"
E = np.array(read_excel_data(filename, sheet_name)[1+ind_0:1+ind_f,1].tolist())
Ta = np.array(read_excel_data(filename, sheet_name)[1+ind_0:1+ind_f,2].tolist())

#%% On passe au pas de temps de la minute
v_tps_mn = np.arange(v_tps_h[0],v_tps_h[-1], 1)

Pd_mn = np.interp(v_tps_mn, v_tps_h, Pd)
E_mn = np.interp(v_tps_mn, v_tps_h, E)
Ta_mn = np.interp(v_tps_mn, v_tps_h, Ta)

del E, Hd, Pd, v_tps_h, Ta
#%% Production PV
Ppv = Mod_PVprod(Ta_mn, E_mn, Ppv_nom)

In [29]:
def lissage_index_variations_rapides(P_grid):
    """Indice lissage basé sur variations rapides: mean((P(t)-P(t-1))^2)."""
    dP = np.diff(P_grid)
    return np.mean(dP**2)

def mean_equivalent_cycles(E_throughput_kWh, capa_bat_kWh):
    """
    Cycles équivalents par VE (proxy):
      cycles_i = throughput_i / (2 * C)
    donde throughput_i = sum(|ΔE|) (carga + descarga).
    """
    cycles = E_throughput_kWh / (2.0 * capa_bat_kWh)
    return float(np.mean(cycles))

In [30]:
# -----------------------------
# Simulation bidirectionnelle
# -----------------------------
def simulate_bidirectional(
    cap_factor_weekday=0.5,
    k_need_weekday=0.5,
    soc_reserve=0.30,
    peak_threshold_W=300_000.0,
    seed=174
):
    """
    revient:
      P_ev (W, signed)  -> + charge, - décharge
      metrics (dict)    -> lissage, cycles, peak_before/after, energy_in/out
      Pd_mn, Ppv, v_tps_mn, Nj, N_ev, capa_bat, d_autonomie, vmoy, P1_ch, kW2W
    """

    # ----- constantes / params
    dt_h = 1.0 / 60.0         # 1 min
    soc_min = 0.10
    soc_max = 1.00
    soc_reserve = float(np.clip(soc_reserve, soc_min, soc_max))

    cons_kWh_km = capa_bat / d_autonomie     # kWh/km
    Pmax_W = P1_ch * kW2W                    # 22 kW por véhicule (W)
    rng = np.random.default_rng(seed)

    # ----- arrays
    T = len(Pd_mn)
    P_ev = np.zeros(T)  # signed W

    # throughput énergie pour proxy de cycles
    E_throughput = np.zeros(N_ev)  # kWh (sum |ΔE|)

    # état de la batterie (votre hypothèse : tout le monde commence à 100 %)
    E_bat = np.ones(N_ev) * (soc_max * capa_bat)  # kWh

    # minuit environ (votre série commence à 1h => -60)
    t0_midnight = v_tps_mn[0] - 60

    def is_weekend(day_from_start):
        # tu notebook: 7 días; 0..4 weekdays y 5..6 weekend
        return day_from_start >= 5

    # énergie de base sans VE
    P_base = Pd_mn - Ppv

    # =========================
    # Loop jours
    # =========================
    for d in range(Nj):
        day_midnight = t0_midnight + d * 1440
        i_day0 = np.searchsorted(v_tps_mn, day_midnight, side="left")
        i_day1 = np.searchsorted(v_tps_mn, day_midnight + 1440, side="left")
        if i_day1 <= i_day0:
            continue

        weekend = is_weekend(d)
        cap_factor = 1.0 if weekend else cap_factor_weekday

        # --------------------------------------------------------
        # 1) Mobilité + fenêtres de connexion (travail et maison)
        # --------------------------------------------------------
        if weekend:
            # finde: sin commute, connectés toute la journée
            E_day = np.zeros(N_ev)  # kWh dépensés
            # connexion pour charger (toute la journée)
            work_arr_idx = np.full(N_ev, i_day0)
            work_dep_idx = np.full(N_ev, i_day1)
            # connexion maison : aussi toute la journée (équivalent)
            home_arr_idx = np.full(N_ev, i_day0)
            home_dep_idx = np.full(N_ev, i_day1)

            # next-day dep (pour compléter la logique de « maison jusqu’à sortie »)
            next_home_dep_idx = np.full(N_ev, min(i_day1, T))

        else:
            # weekdays: randomisation identique à celle dans votre sens unique
            hdep_home = rng.triangular(7.5, 8.5, 9.5, size=N_ev)
            d_traj = rng.triangular(10.0, 30.0, 50.0, size=N_ev)

            # temps de trajet (en heures). Sur votre ordinateur portable : vmoy*60 = km/h
            t_traj_h = (d_traj / (vmoy * 60.0))

            harr_work = hdep_home + t_traj_h
            hdep_work = rng.triangular(17.0, 18.0, 19.0, size=N_ev)
            hdep_work = np.maximum(hdep_work, harr_work + 0.25)

            harr_home = hdep_work + t_traj_h

            # énergie quotidienne (aller-retour)
            E_day = 2.0 * d_traj * cons_kWh_km

            # indices connexion travail
            work_arr_min = day_midnight + np.round(harr_work * 60.0).astype(int)
            work_dep_min = day_midnight + np.round(hdep_work * 60.0).astype(int)
            work_arr_idx = np.searchsorted(v_tps_mn, work_arr_min, side="left")
            work_dep_idx = np.searchsorted(v_tps_mn, work_dep_min, side="left")

            # taux de connexion maison (arrivée à la maison le même jour)
            home_arr_min = day_midnight + np.round(harr_home * 60.0).astype(int)
            home_arr_idx = np.searchsorted(v_tps_mn, home_arr_min, side="left")

            # départ à la maison le lendemain (nous avons utilisé hdep_home du "lendemain")
            # pour ne pas refaire l’échantillonnage deux fois, nous prenons un triangle égal et cohérent
            hdep_home_next = rng.triangular(7.5, 8.5, 9.5, size=N_ev)
            next_midnight = day_midnight + 1440

            home_dep_min_next = next_midnight + np.round(hdep_home_next * 60.0).astype(int)
            next_home_dep_idx = np.searchsorted(v_tps_mn, home_dep_min_next, side="left")

            # pour le jour actuel, la maison est connectée jusqu’à min(i_day1, sortie réelle si elle tombe plus tôt)
            home_dep_idx = np.minimum(next_home_dep_idx, i_day1)

        # appliquer les dépenses de mobilité (respecte le SOC min/max)
        E_bat -= E_day
        E_bat = np.clip(E_bat, soc_min * capa_bat, soc_max * capa_bat)

        # --------------------------------------------------------
        # 2) Objectif de charge (même logique que votre ordinateur portable)
        # --------------------------------------------------------
        if weekend:
            # finde: volver a 100%
            E_need = (soc_max * capa_bat) - E_bat
        else:
            # weekdays: récupérer fraction de la dépense quotidienne
            E_need = k_need_weekday * E_day

        E_need = np.maximum(E_need, 0.0)
        E_rem = E_need.copy()

        # --------------------------------------------------------
        # 3) Simulation minute à minute (charge PV + décharge de pointe nocturne)
        # --------------------------------------------------------
        for t in range(i_day0, i_day1):
            pv_W = float(max(Ppv[t], 0.0))
            pd_W = float(Pd_mn[t])

            # ============ A) CHARGEMENT (uniquement lorsqu’il y a un PV > 0)
            # charge autorisée lorsqu’ils sont connectés au travail ou le week-end toute la journée.
            # (à la maison : selon tes indications, ils ne sont PAS chargés ; ils sont seulement statiques pour le téléchargement)
            if pv_W > 0.0:
                P_total_allowed = cap_factor * pv_W
                if P_total_allowed > 0.0:
                    plugged_charge = (t >= work_arr_idx) & (t < work_dep_idx) & (E_rem > 1e-9) & (E_bat < soc_max * capa_bat - 1e-9)
                    active = np.where(plugged_charge)[0]
                    if active.size > 0:
                        # puissance requise par VE
                        P_need_W = (E_rem[active] / dt_h) * 1000.0  # W
                        P_cap_i = np.minimum(Pmax_W, P_need_W)

                        # water-filling
                        P_alloc = np.zeros(active.size)
                        remaining = P_total_allowed
                        alive = np.ones(active.size, dtype=bool)

                        while remaining > 1e-6 and alive.any():
                            idx_local = np.where(alive)[0]
                            share = remaining / idx_local.size
                            give = np.minimum(share, P_cap_i[idx_local] - P_alloc[idx_local])
                            P_alloc[idx_local] += give
                            remaining -= give.sum()
                            alive[idx_local] = (P_alloc[idx_local] < P_cap_i[idx_local] - 1e-9)

                        # appliquer la charge
                        P_ev[t] += P_alloc.sum()

                        dE = (P_alloc / 1000.0) * dt_h  # kWh
                        E_rem[active] = np.maximum(E_rem[active] - dE, 0.0)

                        E_bat[active] = np.clip(E_bat[active] + dE, soc_min * capa_bat, soc_max * capa_bat)
                        E_throughput[active] += dE  # |ΔE| en charge

            # ============ B) décharge (uniquement lorsque PV = 0 et Pd dépasse le seuil)
            # règle : identifier les pics domestiques lorsque PV=0 et Pd > peak_threshold_W
            if pv_W <= 0.0 and pd_W > peak_threshold_W:
                # "besoin réseau" : dépassement du seuil
                P_excess = pd_W - peak_threshold_W  # Idéalement à réduire

                # flotte connectée à télécharger :
                # - en weekdays : je travaille s’ils sont encore à la maison depuis leur arrivée jusqu’au départ le lendemain
                # - le week-end : connecté toute la journée
                plugged_dis = (
                    ((t >= work_arr_idx) & (t < work_dep_idx)) |
                    ((t >= home_arr_idx) & (t < home_dep_idx))
                )

                # condition SOC pour permettre le téléchargement (réservation)
                can_dis = plugged_dis & (E_bat > soc_reserve * capa_bat + 1e-9)

                active_dis = np.where(can_dis)[0]
                if active_dis.size > 0 and P_excess > 0.0:
                    # limite totale par matériel (tous identiques)
                    P_total_dis_allowed = min(P_excess, active_dis.size * Pmax_W)

                    # energie téléchargeable par EV sans descendre de soc_reserve
                    E_avail = E_bat[active_dis] - soc_reserve * capa_bat  # kWh
                    P_energy_limit_W = (E_avail / dt_h) * 1000.0          # W (por minuto)

                    P_cap_dis_i = np.minimum(Pmax_W, P_energy_limit_W)
                    P_cap_dis_i = np.maximum(P_cap_dis_i, 0.0)

                    # distribuer la décharge (water-filling)
                    P_alloc_dis = np.zeros(active_dis.size)
                    remaining = P_total_dis_allowed
                    alive = (P_cap_dis_i > 1e-9)

                    while remaining > 1e-6 and alive.any():
                        idx_local = np.where(alive)[0]
                        share = remaining / idx_local.size
                        give = np.minimum(share, P_cap_dis_i[idx_local] - P_alloc_dis[idx_local])
                        P_alloc_dis[idx_local] += give
                        remaining -= give.sum()
                        alive[idx_local] = (P_alloc_dis[idx_local] < P_cap_dis_i[idx_local] - 1e-9)

                    # appliquer la décharge (signe négatif dans P_ev)
                    P_ev[t] -= P_alloc_dis.sum()

                    dE_dis = (P_alloc_dis / 1000.0) * dt_h  # kWh que salen de batería
                    E_bat[active_dis] = np.clip(E_bat[active_dis] - dE_dis, soc_min * capa_bat, soc_max * capa_bat)
                    E_throughput[active_dis] += dE_dis  # |ΔE| en descarga

        # fin loop t
    # =========================
    # Indicateurs globaux
    # =========================
    P_grid = Pd_mn + P_ev - Ppv
    metrics = {
        "lissage": lissage_index_variations_rapides(P_grid),
        "cycles_mean": mean_equivalent_cycles(E_throughput, capa_bat),
        "peak_before_kW": float(np.max(P_base) / 1000.0),
        "peak_after_kW": float(np.max(P_grid) / 1000.0),
        "energy_charge_kWh": float(np.sum(np.maximum(P_ev, 0.0)) / 1000.0 * dt_h),
        "energy_discharge_kWh": float(np.sum(np.maximum(-P_ev, 0.0)) / 1000.0 * dt_h),
    }
    return P_ev, metrics

In [31]:
# -----------------------------
# Tracer un scénario bidirectionnel
# -----------------------------
def plot_scenario_bidirectional(cap_factor=0.5, k_need=0.5, soc_reserve=0.30, peak_threshold_kW=300.0):
    peak_threshold_W = peak_threshold_kW * 1000.0

    P_ev, metrics = simulate_bidirectional(
        cap_factor_weekday=cap_factor,
        k_need_weekday=k_need,
        soc_reserve=soc_reserve,
        peak_threshold_W=peak_threshold_W
    )

    P_grid = -Pd_mn - P_ev + Ppv
    P_base = Pd_mn - Ppv

    fig, ax = plt.subplots(figsize=(18, 5), dpi=120)
    ax.plot(v_tps_mn, Pd_mn, label="$P_d$ (domestique)")
    ax.plot(v_tps_mn, Ppv,   label="$P_{pv}$")
    ax.plot(v_tps_mn, P_ev,  label="$P_{ev}$ (signed)")
    ax.plot(v_tps_mn, P_grid, label="$P_{grid}=P_{pv}-P_d-P_{ev}$")
    ax.axhline(peak_threshold_W, linestyle="--", label="Seuil pic (W)")
    ax.set_xlabel("Temps (min)")
    ax.set_ylabel("Puissance (W)")
    ax.grid(True)
    ax.legend(loc="upper right")
    plt.show()

    print("---- Metrics ----")
    for k, v in metrics.items():
        print(f"{k}: {v}")

In [32]:
# ============================================================
# Sliders (variables de l’étape 6)
# ============================================================
cap_slider = widgets.FloatSlider(
    value=0.50, min=0.00, max=1.50, step=0.05,
    description="cap_factor", continuous_update=False
)

kneed_slider = widgets.FloatSlider(
    value=0.50, min=0.00, max=1.00, step=0.05,
    description="k_need", continuous_update=False
)

socres_slider = widgets.FloatSlider(
    value=0.30, min=0.10, max=1.00, step=0.05,
    description="soc_reserve", continuous_update=False
)

peak_slider = widgets.FloatSlider(
    value=300.0, min=0.0, max=600.0, step=10.0,
    description="peak_kW", continuous_update=False
)

ui = widgets.VBox([cap_slider, kneed_slider, socres_slider, peak_slider])
out = widgets.interactive_output(
    plot_scenario_bidirectional,
    {
        "cap_factor": cap_slider,
        "k_need": kneed_slider,
        "soc_reserve": socres_slider,
        "peak_threshold_kW": peak_slider
    }
)

display(ui, out)

Output()